In [1]:
import requests
import re
from api_keys import CLAUDE_FLARE_2  

# 1. Ustawienia Twojego konta
ACCOUNT_ID = "61d2a89d9b1bbe1cb63e322d0f0d22f3"
API_TOKEN = CLAUDE_FLARE_2

# 2. Endpoint API Cloudflare
url = f"https://api.cloudflare.com/client/v4/accounts/{ACCOUNT_ID}/ai/models/search"
headers = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json"
}

# 3. Funkcja pomocnicza do wyciągania wielkości w miliardach (np. z "llama-3-70b" wyciągnie 70.0)
def get_model_size(model_name):
    match = re.search(r'(\d+(?:\.\d+)?)b\b', model_name.lower())
    if match:
        return float(match.group(1))
    return 0.0  # Modele, które nie mają w nazwie "b" trafią na sam dół listy

# 4. Wykonanie zapytania
response = requests.get(url, headers=headers)

if response.status_code == 200:
    data = response.json()
    models = data.get("result", [])
    
    # Krok A: Filtrujemy tylko modele tekstowe
    text_models = [m for m in models if m['task']['name'] == 'Text Generation']
    
    # Krok B: Sortujemy po naszej funkcji wyciągającej parametry (reverse=True daje malejąco)
    text_models.sort(key=lambda m: get_model_size(m['name']), reverse=True)
    
    print(f"✅ Znaleziono łącznie {len(text_models)} modeli do generowania tekstu.\n")
    print("Oto modele posortowane od największego do najmniejszego:")
    
    for i, model in enumerate(text_models, 1):
        size = get_model_size(model['name'])
        size_str = f"[{size}B parametrów]" if size > 0 else "[nieznany rozmiar]"
        print(f"{i}. {size_str} {model['name']}")
        
else:
    print(f"❌ Wystąpił błąd podczas pobierania modeli: {response.status_code}")
    print(response.text)


✅ Znaleziono łącznie 54 modeli do generowania tekstu.

Oto modele posortowane od największego do najmniejszego:
1. [120.0B parametrów] @cf/openai/gpt-oss-120b
2. [120.0B parametrów] @cf/nvidia/nemotron-3-120b-a12b
3. [70.0B parametrów] @cf/meta/llama-3.3-70b-instruct-fp8-fast
4. [32.0B parametrów] @cf/deepseek-ai/deepseek-r1-distill-qwen-32b
5. [32.0B parametrów] @cf/qwen/qwen2.5-coder-32b-instruct
6. [32.0B parametrów] @cf/qwen/qwq-32b
7. [30.0B parametrów] @cf/qwen/qwen3-30b-a3b-fp8
8. [27.0B parametrów] @cf/aisingapore/gemma-sea-lion-v4-27b-it
9. [26.0B parametrów] @cf/google/gemma-4-26b-a4b-it
10. [24.0B parametrów] @cf/mistralai/mistral-small-3.1-24b-instruct
11. [20.0B parametrów] @cf/openai/gpt-oss-20b
12. [17.0B parametrów] @cf/meta/llama-4-scout-17b-16e-instruct
13. [14.0B parametrów] @cf/qwen/qwen1.5-14b-chat-awq
14. [13.0B parametrów] @hf/thebloke/llama-2-13b-chat-awq
15. [12.0B parametrów] @cf/google/gemma-3-12b-it
16. [11.0B parametrów] @cf/meta/llama-3.2-11b-vision-instru

In [2]:
import os
from langchain_cloudflare import ChatCloudflareWorkersAI
from langchain_core.messages import HumanMessage
from api_keys import CLAUDE_FLARE_2

os.environ["CF_ACCOUNT_ID"] = "61d2a89d9b1bbe1cb63e322d0f0d22f3"
os.environ["CF_AI_API_TOKEN"] = CLAUDE_FLARE_2

PROVIDER = "meta"
MODEL = "llama-3.3-70b-instruct-fp8-fast" 

llm = ChatCloudflareWorkersAI(
    model=f'@cf/{PROVIDER}/{MODEL}',
    temperature=1.0,
    max_tokens=4096,
    max_retries=6,
)

def ask_llm(question):
    try:
        message = HumanMessage(content=question)
        response = llm.invoke([message])
        return response.content
    except Exception as e:
        return f"Błąd: {e}"


In [3]:
import json
import random
with open('articles_topics_processed.json', 'r', encoding='utf-8') as fp:
    topics = json.load(fp)
    random.shuffle(topics)

In [4]:
try:
    generated = os.listdir(MODEL)
    generated = [int(x[:-5]) for x in generated]
except FileNotFoundError:
    generated = []

In [5]:
os.makedirs(MODEL, exist_ok=True)

In [6]:
import json
for a in topics:
    if a['ID'] in generated:
        continue
    pytanie = f"""
    Jesteś autorem artykułów dla popularnych polskich mediów.
    Napisz artykuł na temat podany poniżej. Weź pod uwagę, wszystkie informacje,
    które zaraz Ci podam. Czyli: 
    - Zdarzenie, którego to dotyczy,
    - Temat artykułu,
    - Długość i styl artykułu
    
    Napisz artykuł z podanymi cechami:
    Zdarzenie: {a['Zdarzenie']},
    Temat: {a['Temat']},
    Długość i styl artykułu: {a['Długość']}

    Napisz to w formie
    
    Tytuł: 
    Treść:
    
    Nie podawaj żadnej dodatkowej treści typu: Oto artykuł czy jakiś znaków specjalnych"""

    t = ask_llm(pytanie)
    if 'Tytuł:' not in t:
        raise ValueError()
    frag = t.split('Treść:')
    a['Wygenerowany tytuł'] = frag[0].split('Tytuł: ')[-1]
    a['Wygenerowany tekst'] = frag[1]
    a['Model'] = f"@cf/{PROVIDER}/{MODEL}"
    with open(f"{MODEL}/{a['ID']}.json", "w", encoding='utf-8') as f:
        json.dump(a, f, ensure_ascii=False)
    print(f'{a["ID"]} / {len(topics)}')

ValueError: 